# 02 — Mass Budget

Bottom-up component rollup with a contingency factor and a tornado
sensitivity chart. Parametric components (motors, ESCs, props, battery)
are set in this notebook; hardware-fixed items come from
`airframe_params.yaml`; payload contributions come from
`payload_manifest.yaml`.


In [ ]:
"""Add the repo root to sys.path so `import models` works from any subdir."""
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "models").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

FIG_DIR = REPO_ROOT / "outputs" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

import yaml
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
from models.mass_model import MassBudget

# Parametric components for the baseline X6 configuration.
# Edit these rows when the hardware shortlist changes.
parametric_g = {
    "motors_tarot_5008_340kv_x6":  6 * 158,   # ~158 g each incl. cable
    "esc_t-motor_alpha_60a_x6":    6 * 40,
    "props_18x55_cf_x6":            6 * 30,
    "battery_6s_16ah_li_ion":      1900,      # placeholder; swept in NB05
    "antivib_mount_+_skids":        180,
}

# Active payload bay configuration for this run.
active_payloads = ["global_shutter_rgb", "flir_boson_plus"]

mb = MassBudget.from_yaml(
    REPO_ROOT / "airframe" / "airframe_params.yaml",
    REPO_ROOT / "payload" / "payload_manifest.yaml",
    active_payloads=active_payloads,
    extra_components_g=parametric_g,
)

print(f"Dry mass:        {mb.dry_mass_g:>7.0f} g")
print(f"Contingency:     {mb.contingency_mass_g:>7.0f} g "
      f"({mb.contingency_fraction*100:.0f} %)")
print(f"Total AUW:       {mb.auw_g:>7.0f} g  ({mb.auw_kg:.3f} kg)")


## Component breakdown


In [ ]:
import pandas as pd

table = pd.DataFrame(mb.as_table(), columns=["component", "mass_g", "fraction_of_auw"])
table["fraction_of_auw"] = (table["fraction_of_auw"] * 100).round(1)
table


## Tornado sensitivity

Shows AUW reach if each component mass moves +/- 10 %. Long bars =
candidates for tighter sourcing or aggressive weight reduction.


In [ ]:
rows = mb.tornado(fraction=0.10)
names = [r[0] for r in rows]
lows = np.array([r[1] for r in rows])
highs = np.array([r[2] for r in rows])
baseline = mb.auw_kg

fig, ax = plt.subplots(figsize=(7, 0.35 * len(names) + 1.5))
y = np.arange(len(names))
ax.barh(y, highs - lows, left=lows, color="C0", alpha=0.8)
ax.axvline(baseline, color="k", ls="--", lw=1, label=f"baseline {baseline:.2f} kg")
ax.set_yticks(y)
ax.set_yticklabels(names)
ax.invert_yaxis()
ax.set_xlabel("AUW [kg]")
ax.set_title("Mass tornado — +/- 10 % per component")
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(FIG_DIR / "02_mass_tornado.png", dpi=140)
plt.show()


## Hand-off to propulsion sizing


In [ ]:
mass_summary = {
    "auw_kg": mb.auw_kg,
    "dry_mass_g": mb.dry_mass_g,
    "contingency_fraction": mb.contingency_fraction,
    "active_payloads": active_payloads,
}
mass_summary
